# Results

In [1]:
import duckdb
import pandas as pd
import sys
import spacy
import os
sys.path.append('..')

from src.utils import make_corpus, preprocess_spacy
from src.semantic import build_semantic_index, semantic_search
from src.bm25 import bm25_search, build_bm25

## Build Corpus and Preprocessing

In [2]:
# Read data and drop missing values
c2 = duckdb.connect()
data = c2.execute(f"SELECT * FROM read_parquet('../data/raw/merged.parquet')").df()
data.dropna(subset=['product_title'], inplace=True)

In [3]:
# Extract fields for retrieval
cols = ['product_title', 'main_category', 'store', 'title', 'text']

corpus = make_corpus(df=data, cols=cols, asin="asin")

In [4]:
# preprocess corpus and save it
os.makedirs('data/processed', exist_ok=True)

# if corpus is already processed and saved, pass to save time
corpus_path = "../data/processed/preprocessed_corpus.csv"
if os.path.exists(corpus_path):
    corpus = pd.read_csv(corpus_path)
else:
    nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])
    corpus["text"] = [preprocess_spacy(text) for text in nlp.pipe(corpus["text"])]
    corpus.to_csv(corpus_path)

## Save Indices for BM25 and Embeddings

In [5]:
# BM25 index

pickle_path = "../data/processed/bm25.pkl"
bm25 = build_bm25(pickle_path, corpus)

In [6]:
# Semantic index 
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
semantic_index_path = '../data/processed/embedding.faiss'

if not os.path.exists(semantic_index_path):
    build_semantic_index(corpus, model, semantic_index_path)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Retrieve Results

In [7]:
queries = ["Wet wipes",
           "Bar Soap",
           "Small hair dryer",
           "The best air humidifer with essential oil",
           "mineral sunscreen for babies",
           "hair spray that last more than 6 hours",
           "Best Vitamins or supplements to take for pregnant women",
           "best stainless steel pan for cooking",
           "sunrise lamp that will help me to wake up in the morning",
           "Something to relieve my back pain"]

In [8]:
for q in queries:
    print(f"QUERY: {q}\n")

    print("BM25 top results:")
    display(bm25_search(q, pickle_path, data))

    print("\nSemantic search top results:")
    display(semantic_search(q, semantic_index_path, model, data))

QUERY: Wet wipes

BM25 top results:


,product_title,text,rating,score
9003,Rolhei 75% Ethanol Wet Wipe - 2 Packs of 100 (...,that the wipes are thick and not thin.,5.0,15.452317
6634,Lens Wipes Pre-moistened Eye Glasses Cleaner W...,"I bought these based on the reviews, but they ...",1.0,12.032342
12808,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry...",These were really wipes more for use in medica...,1.0,11.516642
6716,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry...",What we liked most was that it does an excelle...,2.0,11.441593
15207,"Pre-Moistened Lens Cleaning Wipes, Wet and Dry...",Do not buy this item. The wipes are so small t...,1.0,10.531983



Semantic search top results:


,product_title,text,rating,score
460,Pampers Baby Fresh Water Baby Wipes 3X Pop-Top...,Baby wipes sure have improved since I used the...,5.0,0.920217
6634,Lens Wipes Pre-moistened Eye Glasses Cleaner W...,"I bought these based on the reviews, but they ...",1.0,0.876085
2377,Rinse Free Sponge Bath Wipes (30-pack) | Extra...,I was very pleased with these rinse-free bath ...,4.0,0.851884
4990,Wet-it Skrubba New European Scrubby Non-Scratc...,"Love these scrubbers, colorful and work well, ...",5.0,0.849512
230,Rinse Free Sponge Bath Wipes (30-pack) | Extra...,When you can't get in the shower and want to f...,4.0,0.840174


QUERY: Bar Soap

BM25 top results:


...

,product_title,text,rating,score
18743,Verilux Original Natural Spectrum Deluxe Floor...,I was sitting on my bed last night doing some ...,5.0,1.059190
2847,Verilux Original Natural Spectrum Deluxe Floor...,"I bought this as a gift for my mom, it was at ...",4.0,1.023793
16576,Indus Classic Pine Himalayan Salt Crystal Lamp...,This was my first salt lamp. It is very well ...,5.0,1.013871
2796,Naturebright L6060 Per2 Led Daylight Lamp,This functions well as daylight alarm. It's be...,3.0,0.727630
18956,Naturebright L6060 Per2 Led Daylight Lamp,I tried the Phillips previously (see review th...,4.0,0.675205


QUERY: Something to relieve my back pain

BM25 top results:


,product_title,text,rating,score
491,Neck Stretcher Cervical Neck Traction Device O...,First time user. It took me a while a find out...,5.0,10.952530
12649,Shoulder Wrap Gel Ice Hot Cold Pack for Should...,"I use it alot, great product, helps relieve my...",5.0,10.870432
14553,ZSZBACE Posture Corrector Back Brace for Men a...,It help with my back pain,5.0,10.828765
671,Korean Red Ginseng Patch Powerstrip Energy Pai...,These patches really do relieve the pain. They...,5.0,10.687405
6944,Real Time Pain Relief George Foreman's Knockou...,This stuff does exactly what it says it will d...,5.0,10.650396



Semantic search top results:


,product_title,text,rating,score
10646,"REEHUT Foam Roller - (6""x36"") Firm High-Densit...",Is what your back muscles and spine will say!<...,5.0,0.847140
2599,Lower Back Stretcher Spine Board-Back Stretche...,This really gives a nice stretch. I use it on...,5.0,0.810345
12300,Genericb Back Massage Stretcher Arch Magic Mes...,A little to stiff for sore back muscles. need ...,4.0,0.758031
2628,Acupuncture Mat and Pillow Set-Relieve Your St...,Really helped with tension in my shoulders!,5.0,0.709054
4439,Large Heating and Cooling Reusable Wrap by Soo...,This absolutely what I have been looking for t...,5.0,0.699777
